# Exercise 5 — Inspect and save the report

**Learner exercise** · [All exercises](../index.html) · [Setup](../README.md)

## What you’ll learn

- Save the report and rejected rows as Parquet, then read them back to check their values.
- Identify which operations describe work and which actions execute it.

**Core: about 5 minutes.** The same baseline for everyone. [Optional zoom-in](#zoom): about 4 extra minutes; choose it here if the topic interests you.

Complete **Your code**, run the **Check** cells, and open hints when needed. Replace `todo(...)` with your answer. Do not use **Run All** while tasks remain unfinished.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Stop Spark in the previous notebook before closing it. This uses the `create_spark` helper explained in [Exercise 0](00-spark-session.ipynb). Missing earlier work? Use an explicit [catch-up step](../docs/RECOVERY.md).

In [ ]:
import sys
from pathlib import Path

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'lab_support/runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

from lab_support import checks as check
from lab_support.arrival_files import publish_arrival
from lab_support.checks import todo
from lab_support.runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path
from lab_support.workspace import Workspace

workspace = Workspace(solutions=False)
product_key, clean_products, clean_sales, accepted_sales, rejected_sales, enrich_sales, category_totals = workspace.load('product_key', 'clean_products', 'clean_sales', 'accepted_sales', 'rejected_sales', 'enrich_sales', 'category_totals')
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
products = clean_products(raw_products)
cleaned = clean_sales(raw)
accepted = accepted_sales(cleaned)
rejected = rejected_sales(cleaned)
enriched = enrich_sales(accepted, products)
report = category_totals(enriched)
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

---
<a id="exercise-5"></a>
## Your task

**What work have we described, and how can we check the saved result?**

Core budget: about 5 minutes.

Save the category report and rejected rows to the supplied fresh paths. Read the report back as `saved_report` and verify its values. The write/read steps are supplied for the core. Explain which operations actually execute work; the optional section investigates the physical plan.

In [ ]:
REPORT_PATH = spark_path(RUN_ROOT / "report")
REJECTED_PATH = spark_path(RUN_ROOT / "rejected")

### Supplied — write, then read back

Use the default `errorifexists` mode to protect existing outputs. Run the write cell once; after a successful write, rerun just the read/check cells.

In [ ]:
report.write.mode("errorifexists").parquet(REPORT_PATH)
rejected.write.mode("errorifexists").parquet(REJECTED_PATH)

In [ ]:
saved_report = spark.read.parquet(REPORT_PATH)

### Check — supplied

The saved values must match your in-memory report, and all three rejected rows must be saved.

In [ ]:
check.saved(report, saved_report, spark.read.parquet(REJECTED_PATH))
saved_report.orderBy("category").show()

Slide reminders: [Transformations and Actions](https://dannyscodecorner.github.io/mastering-pyspark/#transformations-actions) · [Explain the plan](https://dannyscodecorner.github.io/mastering-pyspark/#explain-plan).

<details>
<summary>Need a nudge? Hint 1</summary>

The DataFrame writer belongs to `.write`; the batch reader belongs to `spark.read`.

</details>

<details>
<summary>A little more help: Hint 2</summary>

Writing is an action. The Python call returns None; the files are its side effect. Keep writes separate from read-back checks.

</details>

If you need to catch up during class, use the explicit [recovery step](../docs/RECOVERY.md#exercise-5). [Worked solution](../solutions/05-save-report.ipynb) — open it separately when you are ready to compare.

## Core complete

For the 60-minute lab, [skip to Save and finish](#finish). To explore this topic further, continue with the optional section below. Later core exercises do not need any of its variables.

<a id="zoom"></a>
## Optional zoom-in · about 4 minutes

These investigations make up the extra depth in a 90-minute session. Choose them independently; keep your working pipeline unchanged.

### Inspect the plan

Run the next cell. Find the join strategy and any `Exchange` operators shown. Explain which earlier operations they relate to. Exact operators can vary; eight rows on a laptop are not a performance benchmark.

Your observations: …

In [ ]:
report.explain(mode="formatted")

### Small checks, deliberate actions

Our helpers use `collect()` and counts on a tiny fixture to make mistakes visible. Do not copy repeated whole-dataset checks into a large production job without considering their cost. A formatted plan explains planned work; it does not prove runtime performance.

The [repeated-work experiment](deeper/plans-and-cache.ipynb) compares plans and explores caching without making timing claims.

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [ ]:
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Next: [Exercise 6 — Process arriving files](06-streaming.ipynb).

Want more on this topic? You can open these now, using the same saved work: [Plans and repeated work](deeper/plans-and-cache.ipynb).

After your attempt, compare the separate [worked solution](../solutions/05-save-report.ipynb).